# Chimeric RNA-Seq Negative Data Pipeline (Updated)

This notebook implements a full pipeline to produce labeled **False Negative** and **False Positive** training data for a chimera detection model.

## 1. Terminology & Strategy

Both datasets generated here will be labeled as **False (0)**, but they serve different purposes:

* **False Negative Candidates (Baseline / Canonical):**
    * **Content:** Real, high-quality human transcripts (from UniProt Swiss-Prot).
    * **Goal:** Teach the model what "normal" RNA looks like so it doesn't flag healthy tissue.
    * **Label:** 0

* **False Positive Candidates (Hard Negatives / Synthetic):**
    * **Content:** Synthetic sequences designed to trick the model using advanced operators: **Reverse Complement**, **Shuffled**, and **Shift-Invariant Random Pairs**.
    * **Goal:** Teach the model to distinguish specific fusion breakpoints from random noise, artifacts, or non-coding strand syntax.
    * **Label:** 0

## 1. Setup & Configuration

We will use **Biopython** to handle sequence processing. Ensure it is installed (`pip install biopython`).

In [1]:
# ==============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ==============================================================================
import os
import gzip
import random
import requests
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Configuration
SEED = 42
random.seed(SEED)

# SCALING TARGET:
# We scan the file until we find 30,000 raw human transcripts.
# With 5 variants per transcript, this will produce ~150k final samples.
TARGET_RAW_COUNT = 15000 

# Output Filenames
FN_OUTPUT_FILE = "false_negative_candidates.fasta"  # Canonical (Normal DNA)
FP_OUTPUT_FILE = "false_positive_candidates.fasta"  # Synthetic (Hard Decoys)

# --- THE FIX: USE ENSEMBL CDS (Coding DNA) INSTEAD OF UNIPROT PROTEIN ---
# This file contains the coding sequences (exons stitched together) for all human isoforms.
# It is pure DNA (A, C, G, T).
ENSEMBL_URL = "http://ftp.ensembl.org/pub/current_fasta/homo_sapiens/cds/Homo_sapiens.GRCh38.cds.all.fa.gz"
LOCAL_GZ_FILE = "Homo_sapiens.GRCh38.cds.all.fa.gz"

print(f"✅ Setup Complete. Raw Input Target: {TARGET_RAW_COUNT}")

✅ Setup Complete. Raw Input Target: 15000


## 2. Data Downloader

This step downloads the official UniProt Swiss-Prot database (gzipped) if it doesn't already exist locally.

In [2]:
# ==============================================================================
# CELL 2: DOWNLOAD HUMAN CDS (TRUE DNA)
# ==============================================================================
# We use Ensembl's 'cds.all' file which contains the coding sequences for all isoforms.
ENSEMBL_URL = "http://ftp.ensembl.org/pub/current_fasta/homo_sapiens/cds/Homo_sapiens.GRCh38.cds.all.fa.gz"
local_cds_file = "Homo_sapiens.GRCh38.cds.all.fa.gz"

def download_ensembl_cds(url, local_path):
    if os.path.exists(local_path):
        print(f"✅ Found existing file: {local_path}")
        return
    print(f"⬇️ Downloading Human CDS from {url}...")
    response = requests.get(url, stream=True)
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024*1024):
            if chunk: f.write(chunk)
    print("✅ Download Complete.")

download_ensembl_cds(ENSEMBL_URL, local_cds_file)

✅ Found existing file: Homo_sapiens.GRCh38.cds.all.fa.gz


In [3]:
# ==============================================================================
# CELL 3: LOAD CANONICAL DNA SEQUENCES
# ==============================================================================
def load_canonical_sequences(gz_path, limit):
    records = []
    print(f"📖 Parsing FASTA to find {limit} Human transcripts...")
    
    with gzip.open(gz_path, 'rt') as handle:
        for record in SeqIO.parse(handle, "fasta"):
            # Ensembl headers look like: >ENST00000632684.1 cds chromosome:GRCh38:...
            # We trust these are DNA because they come from the 'cds' directory.
            
            # 1. Length Filter (FuseLens Context Window)
            # We need sequences long enough to be meaningful but fitting in 32k
            if 500 < len(record.seq) < 30000:
                
                # 2. Validity Check (No Protein characters)
                # Ensure it only contains A, C, G, T, N
                if set(str(record.seq).upper()).issubset(set("ACGTN")):
                    records.append(record)
                    if len(records) >= limit:
                        break
                        
    print(f"✅ Loaded {len(records)} verified DNA transcripts.")
    return records

# Load Data
real_transcripts = load_canonical_sequences(local_cds_file, limit=TARGET_RAW_COUNT)

📖 Parsing FASTA to find 15000 Human transcripts...
✅ Loaded 15000 verified DNA transcripts.


## 4. Load Canonical Sequences

We parse the downloaded file, filtering specifically for human sequences (*Homo sapiens*) to create our "Real" baseline.

In [4]:
# ==============================================================================
# CELL 4: GENERATE NEGATIVE DATASETS (Hard-Negative Strategy)
# ==============================================================================
# Strategies:
# 1. Canonical: The real gene (Label 0)
# 2. RevComp: The reverse complement (Label 0) -> Teaches strand awareness
# 3. Reverse: Reversed string, no complement (Label 0) -> Breaks grammar but keeps stats
# 4. Circular Permutation (Jitter): Rotated gene (Label 0) -> Enforces shift invariance

fn_records = [] # Canonical (Normal)
fp_records = [] # Synthetic (Hard Decoys)

strategies = ['rev_comp', 'reverse_only', 'circular_permute']

print(f"\n🚀 Starting Generation Pipeline on {len(real_transcripts)} transcripts...")

for record in real_transcripts:
    seq_str = str(record.seq).upper()
    L = len(seq_str)
    
    # --- A. Canonical Negative (1 per transcript) ---
    fn_records.append(SeqRecord(
        Seq(seq_str),
        id=f"NEG_CANONICAL_{record.id}",
        description="Label:0 source:Ensembl_CDS"
    ))

    # --- B. Synthetic Hard Negatives (3 per transcript) ---
    for strat in strategies:
        
        # 1. Reverse Complement (Wrong Strand)
        if strat == 'rev_comp':
            # Biopython's reverse_complement works correctly on DNA
            rc_seq = str(Seq(seq_str).reverse_complement())
            fp_records.append(SeqRecord(
                Seq(rc_seq),
                id=f"NEG_SYNTH_RC_{record.id}",
                description="Label:0 type:rev_comp"
            ))
            
        # 2. Reverse Only (Breaks Grammar, Keeps Counts)
        elif strat == 'reverse_only':
            rev_seq = seq_str[::-1]
            fp_records.append(SeqRecord(
                Seq(rev_seq),
                id=f"NEG_SYNTH_REV_{record.id}",
                description="Label:0 type:reverse_only"
            ))

        # 3. Circular Permutation (The J-Operator)
        # s[k:] + s[:k] where k ~ Uniform(0, L)
        # Forces the model to look at the whole sequence, not just the middle.
        elif strat == 'circular_permute':
            if L > 200:
                k = random.randint(50, L - 50) 
                permuted_seq = seq_str[k:] + seq_str[:k]
                fp_records.append(SeqRecord(
                    Seq(permuted_seq),
                    id=f"NEG_SYNTH_CIRC_{record.id}_k{k}",
                    description="Label:0 type:circular_permutation"
                ))

# --- Summary ---
total_neg = len(fn_records) + len(fp_records)
print(f"\n✅ GENERATION COMPLETE")
print(f"   ├── Canonical Negatives: {len(fn_records)}")
print(f"   ├── Synthetic Negatives: {len(fp_records)}")
print(f"   └── TOTAL NEGATIVE SET:  {total_neg} samples")


🚀 Starting Generation Pipeline on 15000 transcripts...

✅ GENERATION COMPLETE
   ├── Canonical Negatives: 15000
   ├── Synthetic Negatives: 45000
   └── TOTAL NEGATIVE SET:  60000 samples


## 5. Save to FASTA

Write the records to disk.

In [5]:
# ==============================================================================
# CELL 5: SAVE TO FASTA
# ==============================================================================
print(f"Writing to {FN_OUTPUT_FILE}...")
SeqIO.write(fn_records, FN_OUTPUT_FILE, "fasta")

print(f"Writing to {FP_OUTPUT_FILE}...")
SeqIO.write(fp_records, FP_OUTPUT_FILE, "fasta")

print("\n🎉 FILES READY FOR TRAINING.")
print(f"   1. {os.path.abspath(FN_OUTPUT_FILE)}")
print(f"   2. {os.path.abspath(FP_OUTPUT_FILE)}")

Writing to false_negative_candidates.fasta...
Writing to false_positive_candidates.fasta...

🎉 FILES READY FOR TRAINING.
   1. /home/akp1/GeneAI/false_negative_candidates.fasta
   2. /home/akp1/GeneAI/false_positive_candidates.fasta
